# Barranquilla MOPAC: QM/MM más rápido y el origen de la deriva de energía

Dos preguntas, en una sesión de A100 con el commit de los cambios:

1. **Tiempo.** En la tripsina–BPTI con MOPAC en la GPU, un paso (~0,13 s) se iba en: largo alcance QM–MM en
   Python (~43 ms), el resto de la fuerza en Python (~20 ms), MOPAC (~55 ms; de ellos ~19 ms en dos bucles de un
   solo hilo sobre las cargas MM). Ahora esos bucles usan OpenMP y el largo alcance corre en kernels CUDA propios.
2. **Deriva.** En la chignolina (Mac, plataforma Reference) la parte no conservativa de la fuerza QM/MM fue
   −4,4, −0,5 y −0,15 kJ/mol/ps con tolerancias SCF 0,01, 0,001 y 0,0001. Aquí: la tripsina en CUDA, y la deriva
   del sistema sin región QM (+5,5 kJ/mol/ps antes) con las precisiones `single`, `mixed` y `double` de OpenMM.

| Bloque | Qué hace | Tiempo (A100, estimado) |
|---|---|---|
| Compilar | MOPAC con OpenBLAS | ~10 min |
| A | largo alcance nuevo contra el anterior, sin MOPAC | ~3 min |
| B | dinámica QM/MM de la tripsina: nuevo, ruta Python anterior, perfil de MOPAC | ~8 min |
| C | deriva: MM sola en tres precisiones (20 ps), QM/MM con tres tolerancias y una en `single` (5 ps) | ~40 min |

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU A100*, y luego *Ejecutar todo*. Si Colab
asigna otra GPU, la celda 2 se detiene. Los resultados se guardan en Google Drive
(`MyDrive/mopac_paper_bench/RESULTS_NAME`); si la sesión se corta, vuelva a ejecutar todo y lo terminado se salta.
La última celda descarga un zip.

## 1. Configuración

In [ ]:
COMMIT = 'a6a4992e'            # OpenMP en las cargas MM, kernels CUDA del largo alcance, nve_drift.py
RESULTS_NAME = 'qmmm_speed_drift_a100'
USE_DRIVE = True
SEED = 11
MM_PS = 20.0                   # equilibrado MM de las corridas del bloque B (ps)
TOLS = [0.0001, 0.001, 0.01]   # tolerancias SCF del bloque C (de la más estricta a la del artículo)
NVE_MM = 20000                 # pasos NVE de 1 fs, MM sola
NVE_QM = 5000                  # pasos NVE de 1 fs, QM/MM
QM_SINGLE = True               # además: QM/MM con la tolerancia más estricta en precisión single
QUICK = False                  # True: 20 + 200 pasos (prueba del notebook)
REQUIRE_GPU = 'A100'           # None: aceptar cualquier GPU

## 2. GPU, CPU y carpeta de resultados

In [ ]:
import os, re, subprocess, sys, time, shutil, json, gzip
from pathlib import Path

def run(cmd, check=True, env=None, log=None):
    # Run a shell command and stream its output to the cell (and to a log file)
    print('$', cmd, flush=True)
    full = dict(os.environ); full.update(env or {})
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=full)
    fh = open(log, 'w') if log else None
    for line in p.stdout:
        print(line, end='', flush=True)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if check and p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

run('nvidia-smi')
gpu = subprocess.run('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip()
cpu = subprocess.run("lscpu | grep 'Model name' | sed 's/.*: *//'", shell=True, capture_output=True,
                     text=True).stdout.strip()
NCPU = os.cpu_count()
print(f'GPU: {gpu}\nCPU: {cpu}, {NCPU} núcleos lógicos')
CC = gpu.split(',')[-1].strip().replace('.', '')
if REQUIRE_GPU and REQUIRE_GPU not in gpu:
    raise RuntimeError(f'Esta sesión tiene {gpu}, no una {REQUIRE_GPU}. Cambie el tipo de entorno a {REQUIRE_GPU} '
                     f'(o ponga REQUIRE_GPU = None) y vuelva a ejecutar todo.')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/mopac_paper_bench') / RESULTS_NAME
else:
    OUT = Path('/content/mopac_paper_bench') / RESULTS_NAME
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'env.json').write_text(json.dumps({'gpu': gpu, 'cpu': cpu, 'ncpu': NCPU, 'commit': COMMIT}, indent=1))
print('resultados en', OUT)

## 3. Compilar MOPAC (biblioteca) con OpenBLAS

~10 minutos.

In [ ]:
SRC, BUILD = Path('/content/mopac_gpu'), Path('/content/build')
run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq gfortran ninja-build '
    'libopenblas-dev liblapack-dev > /dev/null')
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
shutil.rmtree(SRC, ignore_errors=True); shutil.rmtree(BUILD, ignore_errors=True)
run(f'git clone -q https://github.com/juvenalyosa/mopac_gpu.git {SRC}')
run(f'git -C {SRC} checkout -q {COMMIT}')
run(f'git -C {SRC} log -1 --format="%h %ad %s" --date=short')
head = subprocess.run(f'git -C {SRC} rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
if not head.startswith(COMMIT):
    raise SystemExit(f'HEAD {head} no es COMMIT {COMMIT}')
stamp = OUT / 'commit.txt'
if stamp.exists() and not head.startswith(stamp.read_text().strip()[:8]):
    raise SystemExit(f'{OUT} tiene resultados de {stamp.read_text().strip()}; use otro RESULTS_NAME')
stamp.write_text(head + '\n')
t0 = time.time()
run(f'cmake -S {SRC} -B {BUILD} -G Ninja -DGPU=ON -DTESTS=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_ARCHS={CC} '
    f'-DENABLE_MKL=OFF -DBLA_VENDOR=OpenBLAS > /content/cmake.log 2>&1 || (tail -40 /content/cmake.log; exit 1)')
run(f'cmake --build {BUILD} --target mopac --parallel {NCPU} > /content/build.log 2>&1 '
    f'|| (grep -i -B2 -A6 error /content/build.log | head -80; exit 1)')
LIBMOPAC = next(BUILD.glob('libmopac.so*'))
print(f'compilado en {time.time() - t0:.0f} s: {LIBMOPAC}')

## 4. OpenMM y CuPy

In [ ]:
m = re.search(r'release (\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout)
cupy_pkg = f'cupy-cuda{m.group(1) if m else 12}x'
other = [p for p in subprocess.run('pip list 2>/dev/null', shell=True, capture_output=True, text=True).stdout.split()
         if p.startswith('cupy-cuda') and p != cupy_pkg]
if other:
    run('pip uninstall -y -q ' + ' '.join(other), check=False)
run(f"pip install -q 'openmm[cuda12]==8.6.1' {cupy_pkg} 2>&1 | tail -2", check=False)
run(f"{sys.executable} -c \"import openmm as m; print('OpenMM', m.__version__, "
    f"[m.Platform.getPlatform(i).getName() for i in range(m.Platform.getNumPlatforms())])\"", check=False)
run(f"cd {SRC}/scripts/qmmm && {sys.executable} -c \"from qmmm_pme import gpu_available, fused_far_field_available; "
    f"print('QM-PME grid on the GPU (CuPy):', gpu_available(), '; fused kernels:', fused_far_field_available())\"",
    check=False)

## 5. Lanzador con tiempos por llamada

`/content/qmmm_prof.py` ejecuta `run_qmmm_md.py` sin cambiarlo y envuelve cuatro métodos para medir cada llamada:
`MopacQMMMForce.compute` (toda la fuerza QM/MM), `QMSession.step` (MOPAC), `_pme_far_potential` y
`_pme_far_gradient` (PME). El tiempo entre el final de una llamada a la fuerza y el comienzo de la siguiente es el
resto del paso (OpenMM). Los tiempos se guardan en `pytimes.json`.

Modos: `cpu` = `MOPAC_NOGPU=1`; `gpu` = `MOPAC_FORCEGPU=1`. Cada variante añade variables de entorno o opciones.
Con `MOPAC_GPU_PROFILE=1`, MOPAC escribe en su salida (`MOPAC_API_OUTPUT`) el tiempo acumulado por sección después
de cada paso; se guarda el último valor de cada sección en `sections.csv` y la salida completa comprimida.

In [ ]:
PROF = Path('/content/qmmm_prof.py')
PROF.write_text('''
import json, os, runpy, sys, time
script = sys.argv[1]
sys.argv = sys.argv[1:]
sys.path.insert(0, os.path.dirname(os.path.abspath(script)))
import openmm_mopac as om
import mopac_qmmm_api as api
T = {"t0": [], "compute": [], "mopac": [], "pme_far": [], "pme_grad": [], "stack_before": [], "stack_after": [],
     "free_mb_after": [], "preset": None}
rt = None
if os.environ.get("QMMM_STACK_LOG") or os.environ.get("QMMM_STACK_PRESET"):
    try:
        import cupy
        rt = cupy.cuda.runtime
        rt.free(rt.malloc(8))                  # create the primary context (the one MOPAC uses)
        if os.environ.get("QMMM_STACK_PRESET"):
            rt.deviceSetLimit(0, int(os.environ["QMMM_STACK_PRESET"]))   # 0: cudaLimitStackSize
            T["preset"] = rt.deviceGetLimit(0)
        print("stack limit at start:", rt.deviceGetLimit(0), flush=True)
    except Exception as e:
        print("stack log unavailable:", e, flush=True)
        rt = None
def stack_log(key):
    if rt is not None:
        T[key].append(rt.deviceGetLimit(0))
        if key == "stack_after":
            T["free_mb_after"].append(rt.memGetInfo()[0] / 2**20)
def wrap(cls, name, key, stamp=False):
    f = getattr(cls, name)
    def g(self, *a, **k):
        t = time.perf_counter()
        if stamp:
            T["t0"].append(t)
        try:
            return f(self, *a, **k)
        finally:
            T[key].append(time.perf_counter() - t)
    setattr(cls, name, g)
wrap(om.MopacQMMMForce, "compute", "compute", True)
wrap(om.MopacQMMMForce, "_pme_far_potential", "pme_far")
wrap(om.MopacQMMMForce, "_pme_far_gradient", "pme_grad")
wrap(api.QMSession, "step", "mopac")
_step = api.QMSession.step
def _logged(self, *a, **k):
    stack_log("stack_before")
    try:
        return _step(self, *a, **k)
    finally:
        stack_log("stack_after")
api.QMSession.step = _logged
try:
    runpy.run_path(script, run_name="__main__")
finally:
    out = os.environ.get("QMMM_PROF_OUT")
    if out:
        with open(out, "w") as fh:
            json.dump(T, fh)
''')

RX = {
    'system': re.compile(r'system: (\d+) atoms, (\d+) residues, box ([0-9.]+) A'),
    'qm': re.compile(r'QM region: (\d+) atoms \+ (\d+) link atoms, charge (-?\d+); long-range QM-MM electrostatics: (.+)'),
    'nvt': re.compile(r'QM/MM NVT: (\d+) steps x ([0-9.]+) fs in ([0-9.]+) s = ([0-9.]+) s/step, ([0-9.]+) ps/day'),
    'nve': re.compile(r'NVE check (\d+) steps: drift (-?[0-9.]+) kJ/mol, slope of a linear fit (-?[0-9.]+) kJ/mol/ps, '
                      r'largest step ([0-9.]+), rms step ([0-9.]+), rms deviation from the fit ([0-9.]+)'),
    'sec': re.compile(r'\[PROFILE\] MOZYME_SECTION name=(\S+) calls=(\d+) ms=([0-9.]+)'),
}
QMMM = SRC / 'scripts' / 'qmmm' / 'run_qmmm_md.py'

def py_breakdown(T, first, last):
    # medians (ms) of the per-call times over calls first..last-1 (the NVT steps)
    import numpy as np
    c_all, t0 = np.array(T['compute']), np.array(T['t0'])
    c = c_all[first:last]
    # rest of the step: from the end of one force call to the start of the next
    j = np.arange(first, min(last, len(t0) - 1))
    gap = t0[j + 1] - (t0[j] + c_all[j])
    r = {'py_compute_ms': np.median(c) * 1e3 if len(c) else None,
         'py_compute_mean_ms': np.mean(c) * 1e3 if len(c) else None,
         'py_mopac_ms': np.median(T['mopac'][first:last]) * 1e3 if T['mopac'][first:last] else None,
         'py_mopac_mean_ms': np.mean(T['mopac'][first:last]) * 1e3 if T['mopac'][first:last] else None,
         'py_pme_ms': (np.median(np.array(T['pme_far'][first:last]) + np.array(T['pme_grad'][first:last])) * 1e3
                       if T['pme_far'][first:last] else 0.0),
         'py_rest_ms': np.median(gap) * 1e3 if len(gap) else None}
    return {k: (round(float(v), 2) if v is not None else None) for k, v in r.items()}

def qmmm(exp, case, variant, pdb, opts, steps, nve=0, mm_ps=10.0, env_extra=None, seed=SEED):
    mode = variant.split('_')[0]
    env_extra = dict(env_extra or {})
    d = OUT / 'runs' / exp / case / variant
    cmd = (f'{sys.executable} -u {PROF} {QMMM} {pdb} --libmopac {LIBMOPAC} {opts} --platform CUDA '
           f'--steps {steps} --nve {nve} --mm-equil-ps {mm_ps} --seed {seed}')
    key = cmd + '\n' + json.dumps(env_extra, sort_keys=True)
    if (d / 'result.json').exists() and (d / 'cmd.txt').exists() and (d / 'cmd.txt').read_text() == key:
        r = json.loads((d / 'result.json').read_text())
        print(f'[{exp}/{case}] {variant}: ya hecho, {r.get("s_per_step")} s/step')
        return r
    shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)
    (d / 'cmd.txt').write_text(key)
    api_out = Path('/tmp') / f'mopac_api_{exp}_{case}_{variant}.out'
    api_out.unlink(missing_ok=True)
    env = {k: v for k, v in os.environ.items() if k not in ('MOPAC_NOGPU', 'MOPAC_FORCEGPU')}
    env.update(OMP_NUM_THREADS=str(NCPU), OPENBLAS_NUM_THREADS=str(NCPU), MOPAC_GPU_DEBUG='1',
               MOPAC_API_OUTPUT=str(api_out), QMMM_PROF_OUT=str(d / 'pytimes.json'))
    env['MOPAC_NOGPU' if mode == 'cpu' else 'MOPAC_FORCEGPU'] = '1'
    env.update(env_extra)
    env = {k: v for k, v in env.items() if v is not None}       # None: remove the variable
    print(f'[{exp}/{case}] {variant} {env_extra}', flush=True)
    t0 = time.perf_counter()
    with open(d / 'log.txt', 'w') as fh:
        rc = subprocess.run(cmd, shell=True, cwd=d, env=env, stdout=fh, stderr=subprocess.STDOUT).returncode
    text = (d / 'log.txt').read_text(errors='ignore')
    r = {'exp': exp, 'case': case, 'variant': variant, 'mode': mode, 'env': json.dumps(env_extra), 'rc': rc,
         'total_s': round(time.perf_counter() - t0, 1)}
    if (m := RX['system'].search(text)):
        r.update(total_atoms=int(m.group(1)), box_A=float(m.group(3)))
    if (m := RX['qm'].search(text)):
        r.update(qm_atoms=int(m.group(1)), link_atoms=int(m.group(2)), qm_charge=int(m.group(3)), long_range=m.group(4).strip())
    if (m := RX['nvt'].search(text)):
        r.update(steps=int(m.group(1)), dt_fs=float(m.group(2)), md_s=float(m.group(3)),
                 s_per_step=float(m.group(4)), ps_per_day=float(m.group(5)))
    if (m := RX['nve'].search(text)):
        r.update(nve_steps=int(m.group(1)), nve_drift_kj=float(m.group(2)), nve_slope_kj_ps=float(m.group(3)),
                 nve_rms_dev_kj=float(m.group(6)))
    if (m := re.search(r'NVE: energy every (\d+) steps, ([0-9.]+) s = ([0-9.]+) s/step', text)):
        r.update(nve_s=float(m.group(2)), nve_s_per_step=float(m.group(3)))
    if (d / 'pytimes.json').exists():
        T = json.loads((d / 'pytimes.json').read_text())
        r['force_calls'] = len(T['compute'])
        r.update(py_breakdown(T, 2, 2 + steps))          # the first 2 calls: first SCF from the Lewis structure
        if T.get('stack_after'):
            sb, sa, fm = T['stack_before'], T['stack_after'], T['free_mb_after']
            r.update(stack_preset=T.get('preset'), stack_first_before=sb[0], stack_first_after=sa[0],
                     stack_last_after=sa[-1], stack_max=max(sa + sb), free_mb_first=round(fm[0]),
                     free_mb_last=round(fm[-1]), free_mb_min=round(min(fm)))
    lines = api_out.read_text(errors='ignore').splitlines() if api_out.exists() else []
    r['mozyme_gpu_lines'] = sum('[MOZYME GPU' in l for l in lines)
    r['gpu_scf_stdout_lines'] = sum('[MOZYME GPU SCF]' in l for l in text.splitlines())
    sec = {}
    for l in lines:
        if (m := RX['sec'].search(l)):
            sec[m.group(1)] = (int(m.group(2)), float(m.group(3)))
    if sec:
        with open(d / 'sections.csv', 'w') as fh:
            fh.write('section,calls,ms_total\n')
            for k, (c, ms) in sorted(sec.items(), key=lambda x: -x[1][1]):
                fh.write(f'{k},{c},{ms:.3f}\n')
    if api_out.exists():
        with open(api_out, 'rb') as fi, gzip.open(d / 'mopac_api.out.gz', 'wb') as fo:
            shutil.copyfileobj(fi, fo)
        api_out.unlink()
    (d / ('result.json' if rc == 0 and 's_per_step' in r else 'result_failed.json')).write_text(json.dumps(r, indent=1))
    print(f'    {r.get("s_per_step")} s/step; per call (ms, median): force {r.get("py_compute_ms")}, MOPAC '
          f'{r.get("py_mopac_ms")}, PME {r.get("py_pme_ms")}, rest {r.get("py_rest_ms")}; [MOZYME GPU] lines '
          f'{r["mozyme_gpu_lines"]}, rc {rc}, {r["total_s"]:.0f} s', flush=True)
    if rc != 0:
        print('\n'.join(text.splitlines()[-10:]))
    return r

def get_pdb(code):
    import urllib.request
    f = OUT / 'runs' / '_pdb' / f'{code}.pdb'
    f.parent.mkdir(parents=True, exist_ok=True)
    if not f.exists():
        urllib.request.urlretrieve(f'https://files.rcsb.org/download/{code}.pdb', f)
    return f

CHIG = SRC / 'tests' / 'data' / 'mozyme_gpu_reference' / '1UAO_opt.pdb'
ENZ = ('--qm-residues E:57:sc,E:102:sc,E:195:sc,I:15,I:16 --variants E:57=HID --padding 1.0 --ron 10 --roff 12 '
       '--distances E:195:OG-I:15:C,E:57:NE2-E:195:OG,E:57:HD1-E:102:OD2')
ENZ_PDB = get_pdb('2PTC')

## A. El largo alcance nuevo en el sistema real (sin MOPAC)

`/content/check_far.py` construye la tripsina–BPTI solvatada, reemplaza MOPAC por un sustituto instantáneo y:
compara `FarFieldGPU` con `FarFieldNumPy` sobre las mismas entradas (potencial en los átomos QM y todos los
gradientes) y mide el tiempo por llamada de toda la parte Python de la fuerza QM/MM con la ruta anterior
(`QMMM_FAR_FIELD=numpy`) y con la nueva.

In [ ]:
CHECK = Path('/content/check_far.py')
CHECK.write_text('''
import json, os, sys, time
import numpy as np, openmm as mm, openmm.app as app, openmm.unit as u
sys.path.insert(0, sys.argv[1])
import openmm_mopac as om, qmmm_pme as qp
print("gpu_available", qp.gpu_available(), "fused_far_field_available", qp.fused_far_field_available(), flush=True)
class Stub:
    def __init__(self, libpath, nums, x0, charge=0, model=0, tolerance=0.01):
        self.n = len(nums); self.rng = np.random.default_rng(1)
    def step(self, xq, xs, qs, ron, roff, qm_pot=None):
        self.mm_pot = self.rng.standard_normal(len(qs)); self.qm_charges = 0.1 * self.rng.standard_normal(self.n)
        return 0.0, np.zeros((self.n, 3)), np.zeros((len(qs), 3))
    def close(self):
        pass
om.QMSession = Stub
ff = app.ForceField("amber14-all.xml", "amber14/tip3p.xml")
pdb = app.PDBFile(sys.argv[2])
mod = app.Modeller(pdb.topology, pdb.positions)
PROT = {"ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE","LEU","LYS","MET","PHE","PRO","SER","THR","TRP","TYR","VAL"}
mod.delete([r for r in mod.topology.residues() if r.name not in PROT])
mod.delete([x for x in mod.topology.atoms() if x.element is not None and x.element.symbol == "H"])
mod.addHydrogens(ff, pH=7.0, variants=[("HID" if (r.chain.id == "E" and r.id == "57") else None) for r in mod.topology.residues()])
mod.addSolvent(ff, padding=1.0 * u.nanometer, neutralize=True)
top, pos = mod.topology, mod.positions
BB = ("N", "H", "CA", "HA", "C", "O", "OXT", "H1", "H2", "H3", "HA2", "HA3")
want = [(("E", "57"), True), (("E", "102"), True), (("E", "195"), True), (("I", "15"), False), (("I", "16"), False)]
qm = [x.index for x in top.atoms() for (ch, rid), sc in want if x.residue.chain.id == ch and x.residue.id == rid and not (sc and x.name in BB)]
print("atoms", top.getNumAtoms(), "QM", len(qm), flush=True)
caps = []
orig_pot = qp.FarFieldGPU.potential
def cap(self, *args):
    caps.append([np.array(x, copy=True) if isinstance(x, np.ndarray) else x for x in args])
    return orig_pot(self, *args)
qp.FarFieldGPU.potential = cap
res = {"atoms": top.getNumAtoms(), "qm": len(qm)}
for mode in ("numpy", "fused"):
    os.environ["QMMM_FAR_FIELD"] = "numpy" if mode == "numpy" else ""
    system = ff.createSystem(top, nonbondedMethod=app.PME, nonbondedCutoff=1.0 * u.nanometer, constraints=app.HBonds)
    q = om.add_qmmm(system, top, pos, qm, 0, "stub", ron=10.0, roff=12.0)
    ctx = mm.Context(system, mm.VerletIntegrator(0.001), mm.Platform.getPlatformByName("CPU"))
    ctx.setPositions(pos)
    st = ctx.getState(getPositions=True)
    q.compute(st)
    tf = {"pot": 0.0, "grad": 0.0}
    for name in ("_pme_far_potential", "_pme_far_gradient"):
        f0 = getattr(om.MopacQMMMForce, name)
        def w(self, *a, _f=f0, _k=("pot" if "potential" in name else "grad")):
            t = time.perf_counter(); r = _f(self, *a); tf[_k] += time.perf_counter() - t; return r
        setattr(q, name, w.__get__(q))
    n = 20
    t0 = time.perf_counter()
    for _ in range(n):
        q.compute(st)
    dt = (time.perf_counter() - t0) / n
    res[mode] = dict(compute_ms=dt * 1e3, far_potential_ms=tf["pot"] / n * 1e3, far_gradient_ms=tf["grad"] / n * 1e3,
                     far_class=type(q.far).__name__, n_mm=q.last["n_mm"])
    print(mode, res[mode], flush=True)
# same inputs through both implementations
if not caps:
    print("FarFieldGPU was not used (fused kernels unavailable): no comparison", flush=True)
    json.dump(res, open(sys.argv[3], "w"), indent=1)
    sys.exit(0)
xq, xm, qm_all, xs, qs, c, L = caps[-1]
Q = 0.1 * np.random.default_rng(3).standard_normal(len(xq))
out = {}
for gpu in (False, True):
    grid = qp.PMEGrid(L, 0.25, 1.2, 4, gpu=gpu)
    ff_ = (qp.FarFieldGPU if gpu else qp.FarFieldNumPy)(grid, 0.25, 13.0)
    phi = ff_.potential(xq, xm, qm_all, xs, qs, c, L)
    gq, gm, pot = np.zeros((len(xq), 3)), np.zeros((len(xs), 3)), np.zeros(len(xs))
    g_all = ff_.gradient(Q, gq, gm, pot)
    out[gpu] = dict(phi=phi, gq=gq, gm=gm, pot=pot, g_all=g_all)
res["compare"] = {k: dict(max_ref=float(np.abs(out[False][k]).max()), max_diff=float(np.abs(out[False][k] - out[True][k]).max()))
                  for k in out[False]}
for k, v in res["compare"].items():
    print(f"{k:6s} max|NumPy| {v['max_ref']:.4g}  max|GPU - NumPy| {v['max_diff']:.2e}")
json.dump(res, open(sys.argv[3], "w"), indent=1)
''')
dA = OUT / 'runs' / 'A_far'
dA.mkdir(parents=True, exist_ok=True)
if not (dA / 'check.json').exists():
    run(f'{sys.executable} -u {CHECK} {SRC}/scripts/qmmm {ENZ_PDB} {dA}/check.json', log=dA / 'log.txt', check=False)
print(json.dumps(json.loads((dA / 'check.json').read_text()), indent=1) if (dA / 'check.json').exists() else 'falló')

## B. Tiempo de la dinámica QM/MM, tripsina–BPTI

Las opciones de la corrida del artículo (400 pasos de 0,5 fs, PME), con MOPAC en la GPU:
`fused` (todo nuevo), `numpy_far` (la ruta Python anterior del largo alcance, `QMMM_FAR_FIELD=numpy`) y
`fused_prof` (con el perfil por secciones de MOPAC, para ver `dcart_tail` y `compfg_hcore`).

In [ ]:
res = []
for variant, envx in (('gpu_fused', {}), ('gpu_numpy_far', {'QMMM_FAR_FIELD': 'numpy'}),
                      ('gpu_fused_prof', {'MOPAC_GPU_PROFILE': '1'})):
    res.append(qmmm('B_md', 'trypsin_pme', variant, ENZ_PDB, ENZ, 20 if QUICK else 400, mm_ps=10.0 if QUICK else MM_PS,
                    env_extra=envx))
for f in sorted((OUT / 'runs' / 'B_md').rglob('sections.csv')):
    print(f.parent.name); print(''.join(f.read_text().splitlines(True)[:14]))

## C. ¿De dónde viene la deriva? (`scripts/qmmm/nve_drift.py`)

Un estado inicial (MM, 50 ps) para todas las corridas. Cada corrida: 400 pasos NVT de 1 fs y luego NVE (velocity
Verlet, 1 fs). `slope_qm_nonconservative` es la parte de la deriva que viene de que la fuerza QM/MM no sea
exactamente el gradiente de su energía; `slope_rest`, lo demás (MM e integración).

- MM sola (`NVE_MM` pasos) con la precisión `single` (la de las corridas anteriores), `mixed` y `double` de OpenMM.
- QM/MM (`NVE_QM` pasos), precisión `mixed`, tolerancias SCF `TOLS`; y una en `single`.

In [ ]:
ND = SRC / 'scripts' / 'qmmm' / 'nve_drift.py'
dC = OUT / 'runs' / 'C_drift'
prep = dC / 'prep'
if not (prep / 'state.xml').exists():
    prep.mkdir(parents=True, exist_ok=True)
    run(f'{sys.executable} -u {ND} prepare {ENZ_PDB} --out {prep} --padding 1.0 --variants E:57=HID '
        f'--mm-equil-ps {10 if QUICK else 50} --platform CUDA --precision mixed', log=prep / 'log.txt')
QMSEL = '--qm-residues E:57:sc,E:102:sc,E:195:sc,I:15,I:16 --ron 10 --roff 12'
runs = [(f'mm_{p}', f'--mm-only --precision {p}', NVE_MM) for p in ('single', 'mixed', 'double')]
runs += [(f'qm_tol{t:g}_mixed', f'{QMSEL} --tolerance {t} --precision mixed', NVE_QM) for t in TOLS]
if QM_SINGLE:
    runs += [(f'qm_tol{TOLS[-1]:g}_single', f'{QMSEL} --tolerance {TOLS[-1]} --precision single', NVE_QM)]
for name, opts, nve in runs:
    d = dC / name
    cmd = (f'{sys.executable} -u {ND} run --prep {prep} --out {d} --libmopac {LIBMOPAC} {opts} --platform CUDA '
           f'--nvt {20 if QUICK else 400} --nve {200 if QUICK else nve} --dt 1.0 --every 10 --seed 11')
    if (d / 'summary.json').exists() and (d / 'cmd.txt').exists() and (d / 'cmd.txt').read_text() == cmd:
        print(name, 'ya hecho'); continue
    d.mkdir(parents=True, exist_ok=True)
    (d / 'cmd.txt').write_text(cmd)
    env = dict(os.environ, OMP_NUM_THREADS=str(NCPU), OPENBLAS_NUM_THREADS=str(NCPU), MOPAC_FORCEGPU='1')
    t0 = time.time()
    with open(d / 'log.txt', 'w') as fh:
        rc = subprocess.run(cmd, shell=True, env=env, stdout=fh, stderr=subprocess.STDOUT).returncode
    print(name, 'rc', rc, f'{time.time() - t0:.0f} s', (d / 'log.txt').read_text().strip().splitlines()[-1][:300])

## D. Resultados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
rows = []
for f in sorted((OUT / 'runs').rglob('result*.json')):
    r = json.loads(f.read_text()); r['ok'] = f.name == 'result.json'; rows.append(r)
md_df = pd.DataFrame(rows)
md_df.to_csv(OUT / 'qmmm_speed.csv', index=False)
display(md_df[[c for c in ['exp', 'variant', 's_per_step', 'ps_per_day', 'py_compute_ms', 'py_mopac_ms', 'py_pme_ms',
                           'py_rest_ms', 'mozyme_gpu_lines', 'ok'] if c in md_df]])
dr = []
for f in sorted((OUT / 'runs' / 'C_drift').glob('*/summary.json')):
    s = json.loads(f.read_text()); s['name'] = f.parent.name; dr.append(s)
dr_df = pd.DataFrame(dr)
dr_df.to_csv(OUT / 'qmmm_drift.csv', index=False)
display(dr_df[[c for c in ['name', 'slope_total', 'slope_total_se', 'slope_qm_nonconservative', 'slope_qm_nonconservative_se',
                           'slope_rest', 'slope_rest_se', 'rms_dev', 'nvt_s_per_step', 'nve_s_per_step', 'atoms'] if c in dr_df]])
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for f in sorted((OUT / 'runs' / 'C_drift').glob('*/nve.txt')):
    r = np.loadtxt(f)
    ax[0].plot(r[:, 0], r[:, 1] - r[0, 1], lw=0.7, label=f.parent.name)
    if 'qm' in f.parent.name:
        ax[1].plot(r[:, 0], (r[:, 4] - r[0, 4]) + (r[:, 5] - r[0, 5]), lw=0.8, label=f.parent.name)
ax[0].set_title('E total − E(0)'); ax[1].set_title('parte no conservativa de la fuerza QM/MM')
for a_ in ax:
    a_.set_xlabel('ps'); a_.set_ylabel('kJ/mol'); a_.legend(fontsize=7); a_.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(OUT / 'qmmm_drift.png', dpi=150); plt.show()

## E. Descargar

In [ ]:
tmp = Path('/content/_qmmm_speed') / RESULTS_NAME
shutil.rmtree(tmp.parent, ignore_errors=True)
tmp.mkdir(parents=True)
for f in ('qmmm_speed.csv', 'qmmm_drift.csv', 'qmmm_drift.png', 'env.json', 'commit.txt'):
    if (OUT / f).exists():
        shutil.copy2(OUT / f, tmp / f)
for pat in ('*.json', 'log.txt', 'cmd.txt', 'sections.csv', 'nve.txt', 'mopac_api.out.gz'):
    for f in (OUT / 'runs').rglob(pat):
        if '_pdb' in f.parts or 'prep' in f.parts and f.name != 'log.txt':
            continue
        dst = tmp / 'runs' / f.relative_to(OUT / 'runs')
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dst)
z = shutil.make_archive(f'/content/{RESULTS_NAME}', 'zip', tmp.parent)
print(z, f'{os.path.getsize(z) / 1e6:.1f} MB')
from google.colab import files
files.download(z)